# Setup and Data Preparation

In [1]:
import pyterrier as pt
import pyt_splade
import os
import torch
import warnings

warnings.filterwarnings("ignore", message="User provided device_type of 'cuda'")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")

torch.manual_seed(26)  # for reproducibility


def detect_device():
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():
        return "mps"
    else:
        return "cpu"


device = detect_device()
print(f"Using device: {device}")

Using device: mps


In [2]:
# Initialize SPLADE model
if device == "mps":
    splade = pyt_splade.Splade(model="naver/splade-cocondenser-ensembledistil", device=device, max_length=256)
else:
    splade = pyt_splade.Splade(model="C:\splade_model", device=device, max_length=256)

# Load Robust04 dataset
dataset = pt.get_dataset("irds:disks45/nocr/trec-robust-2004")

/Users/robert/Code/ir-project/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Create custom corpus iterator
def custom_iter():
    for doc in dataset.get_corpus_iter():
        yield {
            'docno': doc['docno'],
            'text': (doc.get('title', '') + ' ' + doc.get('body', '')).strip()
        }

In [4]:
# Load or create BM25 index
bm25_index_dir = os.path.abspath("./robust04_bm25_index")

if os.path.exists(bm25_index_dir):
    bm25_index_ref = pt.IndexFactory.of(bm25_index_dir)
else:
    bm25_indexer = pt.IterDictIndexer(bm25_index_dir)
    bm25_index_ref = bm25_indexer.index(custom_iter())

Java started (triggered by IndexFactory.of) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


In [5]:
# Load or create SPLADE index
splade_index_dir = os.path.abspath("./robust04_splade_index")

if os.path.exists(splade_index_dir):
    index_ref = pt.IndexFactory.of(splade_index_dir)
else:
    splade_indexer = splade.doc_encoder() >> pt.IterDictIndexer(splade_index_dir)
    index_ref = splade_indexer.index(custom_iter())

# Experiment 1: Baseline SPLADE

In [6]:
from pyterrier.measures import *

splade_retr = splade.query_encoder() >> pt.terrier.Retriever(index_ref, wmodel="Tf")
topics = dataset.get_topics()
topics["query"] = topics["title"]
qrels = dataset.get_qrels()

results = pt.Experiment(
    [splade_retr],
    topics,
    qrels,
    eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
    names=["SPLADE Baseline"]
)

print(results)

There are multiple query fields available: ('title', 'description', 'narrative'). To use with pyterrier, provide variant or modify dataframe to add query column.


              name        AP     R@100   nDCG@10
0  SPLADE Baseline  0.225196  0.373159  0.454902


# Experiment 2: Improved SPLADE with Hybrid Retrieval

In [7]:
splade_retr = splade.query_encoder() >> pt.terrier.Retriever(index_ref, wmodel="Tf")
bm25_retr = pt.terrier.Retriever(bm25_index_ref, wmodel="BM25")

# Hybrid Retrieval with different weights for bm25
hybrid_05 = splade_retr + (0.05 * bm25_retr)
hybrid_10 = splade_retr + (0.10 * bm25_retr)
hybrid_20 = splade_retr + (0.20 * bm25_retr)

In [8]:
from pyterrier.measures import *

topics = dataset.get_topics()
topics["query"] = topics["title"]
qrels = dataset.get_qrels()

experiment = pt.Experiment(
    [splade_retr, bm25_retr, hybrid_05, hybrid_10, hybrid_20],
    topics,
    qrels,
    eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
    names=["SPLADE Only", "BM25 Only", "Hybrid (w=0.05)", "Hybrid (w=0.1)", "Hybrid (w=0.2)"]
)

print(experiment)

There are multiple query fields available: ('title', 'description', 'narrative'). To use with pyterrier, provide variant or modify dataframe to add query column.
              name        AP     R@100   nDCG@10
0      SPLADE Only  0.225196  0.373159  0.454902
1        BM25 Only  0.236869  0.401218  0.424420
2  Hybrid (w=0.05)  0.231576  0.373373  0.455241
3   Hybrid (w=0.1)  0.231805  0.373747  0.455514
4   Hybrid (w=0.2)  0.232332  0.373931  0.456361


# Experiment 3: Reciprocal Rank Fusion

In [ ]:
import pyterrier_alpha as pta

splade_res = splade_retr.transform(topics)
bm25_res = bm25_retr.transform(topics)

rrf = pta.fusion.rr_fusion(splade_res, bm25_res, k=60)

experiment = pt.Experiment(
    [splade_res, bm25_res, hybrid_20, rrf],
    topics,
    qrels,
    eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
    names=["SPLADE", "BM25", "Hybrid (w=0.2)", "RRF"]
)

print(experiment)

# Experiment 4: Hybrid Retrieval Weight Optimisation

In [ ]:
import numpy as np
import pandas as pd
import pyterrier as pt
import pyterrier_alpha as pta

weights = np.arange(0.0, 1.1, 0.1)
results = []

for w in weights:
    hybrid = splade_retr + (w * bm25_retr)
    exp = pt.Experiment(
        [hybrid],
        topics,
        qrels,
        eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
        names=[f'Hybrid (w={w:.1f})']
    )
    results.append(exp)

hybrid_results = pd.concat(results)
print(hybrid_results)

In [ ]:
import numpy as np
import pandas as pd
import pyterrier as pt
import pyterrier_alpha as pta

weights = [1, 2, 3, 5, 8, 10, 15, 20]
results = []

for w in weights:
    hybrid = splade_retr + (w * bm25_retr)
    exp = pt.Experiment(
        [hybrid],
        topics,
        qrels,
        eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
        names=[f'Hybrid (w={w:.1f})']
    )
    results.append(exp)

hybrid_results = pd.concat(results)
print(hybrid_results)

In [ ]:
import numpy as np
import pandas as pd
import pyterrier as pt
import pyterrier_alpha as pta

weights = [20, 30, 40, 50, 75, 100]
results = []

for w in weights:
    hybrid = splade_retr + (w * bm25_retr)
    exp = pt.Experiment(
        [hybrid],
        topics,
        qrels,
        eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
        names=[f'Hybrid (w={w:.1f})']
    )
    results.append(exp)

hybrid_results = pd.concat(results)
print(hybrid_results)

# Experiment 5: Expansion Models

In [ ]:
hybrid_20 = splade_retr + (20 * bm25_retr)

bm25_bo1 = (
        bm25_retr
        >> pt.text.get_text(dataset, ["title", "body"])

        >> pt.apply.text(lambda row: (row['title'] or '') + " " + (row['body'] or ''))

        >> pt.rewrite.Bo1QueryExpansion(bm25_index_ref, fb_docs=3, fb_terms=10)

        >> bm25_retr
)

upgraded_hybrid = splade_retr + (20.0 * bm25_bo1)

pt.Experiment(
    [splade_retr, hybrid_20, upgraded_hybrid],
    topics,
    qrels,
    eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
    names=["SPLADE", "Hybrid (w=20)", "Upgraded Hybrid (w=20 + Bo1)"]
)

In [ ]:
hybrid_20 = splade_retr + (20 * bm25_retr)

bm25_rm3 = (
        bm25_retr
        >> pt.text.get_text(dataset, ["title", "body"])

        >> pt.apply.text(lambda row: (row['title'] or '') + " " + (row['body'] or ''))

        >> pt.rewrite.RM3(bm25_index_ref, fb_docs=10, fb_terms=10)

        >> bm25_retr
)

rm3_hybrid = splade_retr + (20.0 * bm25_rm3)

pt.Experiment(
    [rm3_hybrid],
    topics,
    qrels,
    eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
    names=["Hybrid (w=20 + RM3)"]
)

# Experiment 6: Expansion Model Parameter Optimisation

In [ ]:
lambdas = [0.3, 0.4, 0.5, 0.6, 0.7]
results = []

for lam in lambdas:
    rm3_lambda = (
            bm25_retr
            >> pt.text.get_text(dataset, ["title", "body"])
            >> pt.apply.text(lambda row: (row['title'] or '') + " " + (row['body'] or ''))
            >> pt.rewrite.RM3(bm25_index_ref, fb_docs=10, fb_terms=10, fb_lambda=lam)
            >> bm25_retr
    )

    rm3_final = splade_retr + (20.0 * rm3_lambda)

    results.append(
        pt.Experiment(
            [rm3_final],
            topics,
            qrels,
            eval_metrics=[MAP, nDCG @ 10, Recall @ 100],
            names=[f"RM3 (lambda={lam})"]
        )
    )

hybrid_results = pd.concat(results)
print(hybrid_results)